In [23]:
from pathlib import Path
from collections import Counter
import pandas as pd
import torch


ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA = ROOT / "data"
EEG_DIR = DATA / "pre_processed"
assert DATA.is_dir(), f"Data folder missing: {DATA}"
assert EEG_DIR.is_dir(), f"EEG folder missing: {EEG_DIR}"


# Audit 
for saftey , not part of report or something. Asked ai to make most of the code  but don´t think that matters since it´s just some thing on the side 

## T04 — Audit labels and recording files


1. audit csv -Labels, missng values and duplicate filenaames.

In [24]:

labels = pd.read_csv(DATA / "anon_codex_age_sex.csv")

print("Rows and columns:", labels.shape)
display(labels.head())
display(labels.isna().sum().rename("missing_values"))


print("Columns:", labels.columns.tolist())
filename_col = labels.columns[0]

print("Duplicate filenames:", labels[filename_col].duplicated().sum())
display(labels.describe(include="all"))

label_ids = set(labels[filename_col].astype(str).map(lambda x: Path(x).stem))
files = sorted(EEG_DIR.glob("*.pt"))
file_ids = {path.stem for path in files}

print("Unique label IDs:", len(label_ids))
print("EEG files:", len(files))
print("Labels without EEG files:", sorted(label_ids - file_ids))
print("EEG files without labels:", sorted(file_ids - label_ids))

Rows and columns: (1032, 3)


,Anon. Filename,Quantized Age (years),Quantized Sex
0,2Y32V8.edf,1.487123,Female
1,73K37E.edf,0.375068,Male
2,3FU420.edf,1.841644,Male
3,D9017K.edf,8.061644,Female
4,2QC868.edf,8.880946,Female


Anon. Filename           0
Quantized Age (years)    0
Quantized Sex            0
Name: missing_values, dtype: int64

Columns: ['Anon. Filename', 'Quantized Age (years)', 'Quantized Sex']
Duplicate filenames: 0


,Anon. Filename,Quantized Age (years),Quantized Sex
count,1032,1032.000000,1032
unique,1032,NaN,2
top,2Y32V8.edf,NaN,Male
freq,1,NaN,533
mean,NaN,4.497077,NaN
std,NaN,4.275715,NaN
min,NaN,0.155479,NaN
25%,NaN,1.047671,NaN
50%,NaN,2.806575,NaN
75%,NaN,7.099041,NaN


Unique label IDs: 1032
EEG files: 1032
Labels without EEG files: []
EEG files without labels: []


There are no duplicate filenames, labels without EEG files or EEG files without lages.


2.  audit recording files
    

In [25]:
#pd DataFrame where columns=[ 
#                        "filename", "shape", "dtype", 
#                        "all_finite", "channels", "samples", "duration_seconds", "error" 
#                          ]

SAMPLING_RATE = 64
rows = []

for path in files: 
    row = {"filename": path.name}

    try:
        eeg = torch.load(path, map_location="cpu", weights_only=True)

        if not isinstance(eeg, torch.Tensor):
            raise TypeError(f"Expected tensor, got {type(eeg).__name__}")    

        row["shape"] = tuple(eeg.shape)
        row["dtype"]  =str(eeg.dtype)
        row["all_finite"] = bool(torch.isfinite(eeg).all())

        if eeg.ndim != 2 or eeg.numel() == 0:
            raise ValueError(f"Expected nonempty channels × samples, got {eeg.shape}")
        
        row["channels"] = eeg.shape[0]
        row["samples"] = eeg.shape[1]
        row["duration_seconds"] = eeg.shape[1] / SAMPLING_RATE
        row["error"] = None


    except Exception as error:
        row["error"] = str(error)

rows.append(row)      
audit = pd.DataFrame(rows, columns = ["filename", "shape", "dtype","all_finite", 
                                "channels", "samples", "duration_seconds", "error"])

print("Shape counts:", Counter(audit["shape"].dropna()))
display(audit[["channels", "samples", "duration_seconds"]].describe())


Shape counts: Counter({(20, 57664): 1})


,channels,samples,duration_seconds
count,1.0,1.0,1.0
mean,20.0,57664.0,901.0
std,NaN,NaN,NaN
min,20.0,57664.0,901.0
25%,20.0,57664.0,901.0
50%,20.0,57664.0,901.0
75%,20.0,57664.0,901.0
max,20.0,57664.0,901.0


## T05 — Verify training, validation, and test splits

In [ ]:

splits = {name: torch.load(DATA / f'segments_h1k_{name}.pt', map_location="cpu", weights_only=True) 
                                                            for name in ("train", "val", "test")}


for name, entries in splits.items(): 
    print(f"{name}: {len(entries)} entries")

print("Example training entry:", splits["train"][0])    

split_ids = {}

for name, entries in splits.items():
    ids = [Path(entry[4]).stem for entry in entries]
    split_ids[name] = set(ids)

    print(f"{name}: {len(ids)} entries, " f"{len(split_ids[name])} unique recordings, "
                                f"{len(ids) - len(split_ids[name])} duplicate entries")

overlap_true = []
for first, second in ( ("train", "val"),("train", "test"),("val", "test"),):
    overlap = split_ids[first] & split_ids[second]

    print(f"{first} / {second} overlap ({len(overlap)}):", sorted(overlap))


age_col = "Quantized Age (years)"
sex_col = "Quantized Sex"

split_tables = {}
summary_rows = []

for name, ids in split_ids.items():
    table = labels_by_id[
        labels_by_id["recording_id"].isin(ids)
    ].copy()
    split_tables[name] = table

    summary_rows.append({
        "split": name,
        "unique_recordings": len(ids),
        "matched_label_rows": len(table),
        "age_min": table[age_col].min(),
        "age_median": table[age_col].median(),
        "age_mean": table[age_col].mean(),
        "age_max": table[age_col].max(),
    })

display(pd.DataFrame(summary_rows).set_index("split"))

sex_counts = pd.DataFrame({
    name: table[sex_col].value_counts()
    for name, table in split_tables.items()
}).T.fillna(0).astype(int)

display(sex_counts)


train: 825 entries
val: 103 entries
test: 104 entries
Example training entry: (0, 901, 0, 5.284383562, '8128DH.edf')
train: 825 entries, 825 unique recordings, 0 duplicate entries
val: 103 entries, 103 unique recordings, 0 duplicate entries
test: 104 entries, 104 unique recordings, 0 duplicate entries
train / val overlap (0): []
train / test overlap (0): []
val / test overlap (0): []


,unique_recordings,matched_label_rows,age_min,age_median,age_mean,age_max
split,,,,,,
train,825,825,0.155479,2.975479,4.588638,16.009465
val,103,103,0.155479,2.073973,3.782678,16.009465
test,104,104,0.155479,2.629294,4.478278,16.009465


Quantized Sex,Female,Male
train,394,431
val,49,54
test,56,48
